# Attention, BERT et Retrieval-Augmented Generation

## Solution académique complète des six exercices

Ce notebook combine :

- une analyse théorique des limites des modèles séquentiels ;
- une étude de l’attention dans les Transformers ;
- un système RAG local et fonctionnel ;
- une comparaison expérimentale entre BERT dense et TF-IDF ;
- une comparaison entre génération seule et génération augmentée ;
- une synthèse des avancées et défis futurs du RAG.

L’ensemble fonctionne sans clé API. Les modèles et le dataset sont téléchargés
depuis Hugging Face lors de la première exécution.

**Corpus choisi :** SQuAD, un dataset public de question-réponse fondé sur
des passages issus de Wikipédia.

**Retriever dense :**
`sentence-transformers/msmarco-distilbert-base-v4`, un encodeur DistilBERT
entraîné pour la recherche sémantique.

**Vector store :** FAISS.

**Générateur :** `Qwen/Qwen2.5-0.5B-Instruct`, un petit modèle
autoregressif de type GPT, local et instruction-tuned.

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

1. expliquer pourquoi les anciens modèles séquentiels ont du mal avec les
   dépendances lointaines ;
2. décrire le fonctionnement de l’attention ;
3. construire une chaîne RAG de bout en bout ;
4. mesurer la qualité d’un retriever avec Recall@k ;
5. comparer BERT dense à TF-IDF ;
6. distinguer une erreur de retrieval d’une erreur de génération ;
7. expliquer les avantages et limites du RAG ;
8. identifier les principales directions récentes de recherche.

## Workflow général

### Phase d’indexation

```text
Dataset
   ↓
Documents uniques
   ↓
Découpage en chunks
   ↓
Embeddings BERT
   ↓
Normalisation L2
   ↓
Index FAISS
```

### Phase de question-réponse

```text
Question
   ↓
Embedding BERT de la question
   ↓
Recherche des voisins les plus proches
   ↓
Top-k chunks + métadonnées
   ↓
Prompt : question + contexte
   ↓
Générateur autoregressif
   ↓
Réponse + sources
```

### Principe de diagnostic

Une réponse RAG dépend de deux qualités distinctes :

\[
\text{Qualité RAG}
=
\text{qualité du retrieval}
+
\text{fidélité du générateur}
\]

Il faut donc inspecter les passages récupérés **avant** d’accuser le
générateur.

# 0. Installation et configuration

In [ ]:
%pip install -q \
    "datasets>=3.0,<5.0" \
    "transformers>=4.45,<6.0" \
    "sentence-transformers>=3.0,<6.0" \
    "faiss-cpu>=1.8,<2.0" \
    "scikit-learn>=1.4,<2.0" \
    "accelerate>=1.0,<2.0" \
    "pandas>=2.0,<3.0" \
    "matplotlib>=3.8,<4.0"

In [ ]:
import hashlib
import importlib.metadata as metadata
import math
import random
import re
import textwrap
import warnings
from collections import defaultdict
from dataclasses import dataclass
from time import perf_counter
from typing import Dict, List, Sequence, Tuple

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Appareil :", DEVICE)
print("\nVersions principales :")
for package_name in [
    "datasets",
    "transformers",
    "sentence-transformers",
    "faiss-cpu",
    "scikit-learn",
]:
    try:
        print(f"- {package_name}: {metadata.version(package_name)}")
    except metadata.PackageNotFoundError:
        print(f"- {package_name}: introuvable")

In [ ]:
DATASET_CANDIDATES = ["rajpurkar/squad", "squad"]
DATASET_SPLIT = "validation[:800]"

BERT_RETRIEVER_MODEL_ID = (
    "sentence-transformers/msmarco-distilbert-base-v4"
)
GENERATOR_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

CHUNK_SIZE_WORDS = 160
CHUNK_OVERLAP_WORDS = 35
DEFAULT_K = 3

RETRIEVAL_EVAL_SIZE = 80
GENERATION_EVAL_SIZE = 5

print("Configuration prête.")

# Exercice 1 — Limites des modèles de langage traditionnels

## 1.1 Analyse de la phrase

Phrase étudiée :

> “The scientist, who had been working on the project for years, finally
> made a breakthrough discovery.”

La relation essentielle est :

```text
scientist ─────────────── made ───── discovery
```

Plusieurs groupes de mots séparent `scientist` et `discovery` :

- une proposition relative ;
- une information temporelle ;
- un complément décrivant le projet.

Un ancien modèle récurrent traite les tokens dans l’ordre :

\[
h_t = f(x_t, h_{t-1})
\]

L’information sur `scientist` doit donc traverser plusieurs états cachés
avant d’influencer l’interprétation de `discovery`.

## 1.2 Pourquoi cette dépendance peut être perdue

Dans un RNN simple, les gradients sont multipliés à travers de nombreux pas.
Ils peuvent devenir :

- très petits : *vanishing gradients* ;
- très grands : *exploding gradients*.

Même les LSTM et GRU, conçus pour mieux conserver l’information, doivent
toujours compresser le passé dans une représentation séquentielle limitée.

Le modèle peut donc :

- associer `discovery` à un élément plus proche ;
- sous-estimer que le scientifique est l’auteur de la découverte ;
- produire un résumé incomplet ;
- répondre incorrectement à « Who made the discovery? ».

## 1.3 Conséquences pour les tâches NLP

### Question answering

Le système risque de ne pas relier la question `Who made the discovery?`
au sujet `scientist`.

### Résumé

Il pourrait produire :

> A breakthrough discovery was made.

Ce résumé est grammatical, mais il omet l’acteur principal.

### Extraction de relations

La relation `(scientist, made, discovery)` peut être mal détectée si le
modèle privilégie la proximité linéaire.

## 1.4 Apport de l’attention

L’attention permet à chaque token de comparer directement sa
représentation avec celles des autres tokens :

\[
\text{Attention}(Q,K,V)
=
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

Lorsque le modèle traite `discovery`, il peut attribuer un poids élevé à
`scientist`, même si les deux mots sont éloignés.

La distance dans la phrase ne correspond donc plus à une longue chaîne de
calculs récurrents : le lien peut être modélisé directement.

## 1.5 Illustration numérique simplifiée de l’attention

L’expérience suivante est pédagogique. Les vecteurs sont construits pour
illustrer le calcul, et ne proviennent pas d’un modèle entraîné.

Le token `discovery` joue le rôle de requête. Il compare sa requête aux clés
des autres mots. Une forte similarité avec `scientist` augmente le poids
d’attention correspondant.

In [ ]:
tokens = [
    "The",
    "scientist",
    "working",
    "project",
    "years",
    "made",
    "breakthrough",
    "discovery",
]

keys = np.array(
    [
        [0.0, 0.1, 0.1],
        [1.0, 0.9, 0.1],  # scientist
        [0.2, 0.2, 0.8],
        [0.2, 0.3, 0.7],
        [0.1, 0.2, 0.5],
        [0.8, 0.7, 0.2],  # made
        [0.5, 0.5, 0.3],
        [0.9, 0.8, 0.2],  # discovery
    ],
    dtype=np.float32,
)

discovery_query = np.array([1.0, 0.9, 0.1], dtype=np.float32)

raw_scores = keys @ discovery_query / math.sqrt(keys.shape[1])
attention_weights = np.exp(raw_scores - raw_scores.max())
attention_weights = attention_weights / attention_weights.sum()

attention_df = pd.DataFrame(
    {
        "token": tokens,
        "raw_score": raw_scores,
        "attention_weight": attention_weights,
    }
).sort_values("attention_weight", ascending=False)

display(attention_df)

plt.figure(figsize=(10, 4))
plt.bar(tokens, attention_weights)
plt.title("Poids d’attention simplifiés depuis « discovery »")
plt.ylabel("Poids")
plt.xticks(rotation=35)
plt.show()

> **Précaution scientifique :** un poids d’attention n’est pas
> automatiquement une explication causale complète de la décision du
> modèle. Il montre une interaction interne, mais ne suffit pas toujours à
> expliquer le comportement final.

# Exercice 2 — Impact de l’attention dans les Transformers

## Tâche choisie : question answering

Dans une tâche de question-réponse, le modèle doit rapprocher :

- les mots de la question ;
- les passages susceptibles de contenir la réponse ;
- les indices syntaxiques et sémantiques dispersés dans le contexte.

## 2.1 BERT et l’attention bidirectionnelle

BERT encode chaque token en tenant compte simultanément de son contexte
gauche et droit. Cette bidirectionnalité aide à lever des ambiguïtés.

Exemple :

```text
Question: Who made the discovery?
Context: The scientist, who had worked for years, made a discovery.
```

Le token `who` peut interagir avec `scientist`, tandis que `made` et
`discovery` renforcent la relation recherchée.

## 2.2 Dépendances lointaines

Dans un Transformer, le chemin d’interaction entre deux tokens peut être
direct au sein d’une couche d’auto-attention.

Dans un RNN, l’information doit traverser tous les pas intermédiaires.

Cette différence facilite :

- la coréférence ;
- la désambiguïsation ;
- les relations sujet-verbe-objet ;
- la combinaison de plusieurs indices distants.

## 2.3 Multi-head attention

Plusieurs têtes peuvent apprendre des relations différentes :

- une tête peut suivre la syntaxe ;
- une autre, la coréférence ;
- une autre, les relations sémantiques ;
- une autre, la position.

La représentation finale agrège ces perspectives.

## 2.4 Comparaison historique

Le Transformer original a remplacé les blocs récurrents et convolutifs par
l’attention. Sur WMT 2014 anglais-allemand, le modèle présenté par Vaswani
et al. a obtenu 28,4 BLEU, dépassant les meilleurs résultats précédents de
plus de deux points BLEU, tout en permettant davantage de parallélisme.

Une comparaison littérale avec un « Transformer sans attention » n’est pas
réellement valide : retirer l’attention enlève le mécanisme central qui
définit l’architecture. La comparaison pertinente oppose plutôt :

- les architectures séquentielles récurrentes ;
- les architectures Transformer fondées sur l’attention.

## 2.5 Limites de l’attention

L’attention standard a un coût quadratique par rapport à la longueur :

\[
O(n^2)
\]

Elle peut également :

- se concentrer sur des corrélations superficielles ;
- être sensible au bruit ;
- diluer l’information dans de très longs contextes ;
- exiger beaucoup de mémoire.

# Exercice 3 — Construction d’un système RAG pour le question answering

## 3.1 Pourquoi SQuAD ?

SQuAD fournit :

- une question ;
- un passage Wikipédia ;
- une ou plusieurs réponses de référence ;
- un titre de document.

Il est donc possible de mesurer séparément :

1. si le retriever retrouve le passage d’origine ;
2. si le générateur produit une réponse proche de la référence.

In [ ]:
dataset = None
dataset_name_used = None
loading_errors = []

for candidate in DATASET_CANDIDATES:
    try:
        dataset = load_dataset(candidate, split=DATASET_SPLIT)
        dataset_name_used = candidate
        break
    except Exception as error:
        loading_errors.append((candidate, repr(error)))

if dataset is None:
    raise RuntimeError(
        "Le dataset SQuAD n'a pas pu être chargé. "
        f"Erreurs : {loading_errors}"
    )

print("Dataset chargé :", dataset_name_used)
print("Nombre d'exemples :", len(dataset))
print("Colonnes :", dataset.column_names)

example = dataset[0]
print("\nTitre :", example["title"])
print("Question :", example["question"])
print("Réponse :", example["answers"]["text"][0])
print("Contexte :", example["context"][:500])

## 3.2 Représentation des documents et métadonnées

Nous dédupliquons les contextes : plusieurs questions SQuAD peuvent partager
le même passage.

Chaque document conserve :

- un identifiant stable ;
- son titre ;
- son texte ;
- la liste des identifiants de questions associées.

Les métadonnées permettent ensuite d’évaluer la provenance des résultats.

In [ ]:
@dataclass
class KnowledgeDocument:
    context_id: str
    title: str
    text: str
    question_ids: List[str]


context_registry: Dict[str, KnowledgeDocument] = {}
example_to_context_id: Dict[str, str] = {}

for row in dataset:
    context = row["context"].strip()
    context_hash = hashlib.sha1(
        context.encode("utf-8")
    ).hexdigest()[:16]
    context_id = f"context-{context_hash}"

    example_to_context_id[row["id"]] = context_id

    if context_id not in context_registry:
        context_registry[context_id] = KnowledgeDocument(
            context_id=context_id,
            title=row["title"],
            text=context,
            question_ids=[],
        )

    context_registry[context_id].question_ids.append(row["id"])

knowledge_documents = list(context_registry.values())

print("Exemples SQuAD :", len(dataset))
print("Contextes uniques :", len(knowledge_documents))
print("Premier document :", knowledge_documents[0])

## 3.3 Chunking

Un chunk trop grand peut mélanger plusieurs idées. Un chunk trop court peut
perdre le contexte nécessaire.

Ici, le découpage est réalisé par fenêtres de mots :

- 160 mots par chunk ;
- 35 mots de chevauchement.

L’overlap protège les informations situées à une frontière.

In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    context_id: str
    title: str
    text: str
    start_word: int
    end_word: int


def chunk_document(
    document: KnowledgeDocument,
    chunk_size: int,
    overlap: int,
) -> List[Chunk]:
    words = document.text.split()

    if overlap >= chunk_size:
        raise ValueError("overlap doit être inférieur à chunk_size.")

    step = chunk_size - overlap
    chunks = []

    for start in range(0, len(words), step):
        end = min(start + chunk_size, len(words))
        chunk_words = words[start:end]

        if not chunk_words:
            continue

        chunk_id = f"{document.context_id}-w{start}-{end}"
        chunks.append(
            Chunk(
                chunk_id=chunk_id,
                context_id=document.context_id,
                title=document.title,
                text=" ".join(chunk_words),
                start_word=start,
                end_word=end,
            )
        )

        if end == len(words):
            break

    return chunks


chunks: List[Chunk] = []

for document in knowledge_documents:
    chunks.extend(
        chunk_document(
            document=document,
            chunk_size=CHUNK_SIZE_WORDS,
            overlap=CHUNK_OVERLAP_WORDS,
        )
    )

chunk_texts = [chunk.text for chunk in chunks]

print("Nombre de chunks :", len(chunks))
print("Premier chunk :", chunks[0])
assert len(chunks) > 0
assert all(chunk.text.strip() for chunk in chunks)

## 3.4 Embeddings BERT

Le modèle choisi est un DistilBERT adapté au semantic search.

Contrairement à un embedding de mot statique, sa représentation dépend du
contexte. Le mot `bank` n’a donc pas nécessairement le même vecteur dans
`river bank` et `central bank`.

Les embeddings sont normalisés. Le produit scalaire devient alors
équivalent à la similarité cosinus.

In [ ]:
embedding_start = perf_counter()

bert_embedder = SentenceTransformer(
    BERT_RETRIEVER_MODEL_ID,
    device=DEVICE,
)

chunk_embeddings = bert_embedder.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

embedding_elapsed = perf_counter() - embedding_start

print("Forme des embeddings :", chunk_embeddings.shape)
print(
    f"Embeddings calculés en {embedding_elapsed:.2f} secondes."
)

## 3.5 Index FAISS

`IndexFlatIP` effectue une recherche exacte par produit scalaire.

Puisque les vecteurs sont normalisés :

\[
A \cdot B = \cos(\theta)
\]

Pour un petit corpus pédagogique, une recherche exacte est appropriée.
Pour des millions de vecteurs, on utiliserait plutôt des index
approximatifs ou compressés.

In [ ]:
vector_dimension = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(vector_dimension)
faiss_index.add(chunk_embeddings)

print("Dimension :", vector_dimension)
print("Vecteurs indexés :", faiss_index.ntotal)

## 3.6 Retriever dense

Le retriever :

1. encode la question avec le même modèle ;
2. normalise le vecteur ;
3. recherche les `k` chunks les plus proches ;
4. renvoie les scores et métadonnées.

In [ ]:
def retrieve_dense(question: str, k: int = DEFAULT_K) -> List[Dict]:
    clean_question = question.strip()

    if not clean_question:
        raise ValueError("La question ne peut pas être vide.")

    query_vector = bert_embedder.encode(
        [clean_question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, indices = faiss_index.search(query_vector, k)

    results = []
    for rank, (score, index) in enumerate(
        zip(scores[0], indices[0]),
        start=1,
    ):
        chunk = chunks[int(index)]
        results.append(
            {
                "rank": rank,
                "score": float(score),
                "chunk_index": int(index),
                "chunk_id": chunk.chunk_id,
                "context_id": chunk.context_id,
                "title": chunk.title,
                "text": chunk.text,
                "start_word": chunk.start_word,
                "end_word": chunk.end_word,
            }
        )

    return results


def display_retrieval(results: List[Dict], preview_chars: int = 450):
    rows = []
    for result in results:
        preview = re.sub(r"\s+", " ", result["text"]).strip()
        rows.append(
            {
                "rank": result["rank"],
                "score": round(result["score"], 4),
                "title": result["title"],
                "context_id": result["context_id"],
                "word_range": (
                    f'{result["start_word"]}:{result["end_word"]}'
                ),
                "preview": preview[:preview_chars],
            }
        )

    display(pd.DataFrame(rows))

## 3.7 Sanity check obligatoire du retrieval

Avant de charger le générateur, nous vérifions les passages récupérés.

Cette étape permet de répondre à la question :

> Le contexte contient-il déjà la bonne information ?

Si la réponse est non, changer le générateur ne corrigera probablement pas
le problème.

In [ ]:
sanity_example = dataset[12]
sanity_question = sanity_example["question"]
expected_answer = sanity_example["answers"]["text"][0]
expected_context_id = example_to_context_id[sanity_example["id"]]

sanity_results = retrieve_dense(sanity_question, k=5)

print("Question :", sanity_question)
print("Réponse de référence :", expected_answer)
print("Contexte attendu :", expected_context_id)
print()

display_retrieval(sanity_results)

hit = any(
    result["context_id"] == expected_context_id
    for result in sanity_results
)

print("Passage attendu retrouvé dans le top 5 :", hit)

## 3.8 Chargement du générateur local

Qwen2.5-0.5B-Instruct est un modèle causal autoregressif, c’est-à-dire une
architecture générative de type GPT.

Il est plus adapté aux consignes que GPT-2 tout en restant suffisamment
petit pour une démonstration locale.

Le prompt lui impose :

- d’utiliser uniquement le contexte ;
- de reconnaître l’absence de réponse ;
- de répondre brièvement ;
- de ne pas fabriquer de source.

In [ ]:
generator_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL_ID
)

model_dtype = (
    torch.float16 if torch.cuda.is_available() else torch.float32
)

generator_model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL_ID,
    torch_dtype=model_dtype,
)
generator_model.to(DEVICE)
generator_model.eval()

if generator_tokenizer.pad_token_id is None:
    generator_tokenizer.pad_token_id = (
        generator_tokenizer.eos_token_id
    )

print("Générateur prêt :", GENERATOR_MODEL_ID)

In [ ]:
def generate_with_messages(
    system_message: str,
    user_message: str,
    max_new_tokens: int = 100,
) -> str:
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    prompt = generator_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3072,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = generator_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=generator_tokenizer.pad_token_id,
            eos_token_id=generator_tokenizer.eos_token_id,
        )

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:,
    ]

    return generator_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

## 3.9 Pipeline RAG complet

Le contexte est assemblé avec des identifiants de source explicites.

Le générateur reçoit donc une structure de type :

```text
[Source 1]
passage...

[Source 2]
passage...

Question: ...
```

In [ ]:
RAG_SYSTEM_MESSAGE = (
    "You are a grounded question-answering assistant. "
    "Use only the supplied context. "
    "If the context does not contain the answer, say exactly: "
    "'I do not know based on the retrieved documents.' "
    "Answer concisely and do not invent facts."
)


def answer_with_rag(
    question: str,
    k: int = DEFAULT_K,
    show_sources: bool = True,
) -> Dict:
    retrieval_results = retrieve_dense(question, k=k)

    context_parts = []
    for result in retrieval_results:
        context_parts.append(
            f'[Source {result["rank"]} | {result["title"]}]\n'
            f'{result["text"]}'
        )

    joined_context = "\n\n".join(context_parts)

    user_message = (
        f"Context:\n{joined_context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    answer = generate_with_messages(
        system_message=RAG_SYSTEM_MESSAGE,
        user_message=user_message,
        max_new_tokens=100,
    )

    output = {
        "question": question,
        "answer": answer,
        "sources": retrieval_results,
    }

    print("=" * 100)
    print("QUESTION")
    print(question)
    print("\nRÉPONSE RAG")
    print(answer)

    if show_sources:
        print("\nSOURCES")
        for result in retrieval_results:
            print(
                f'\n[{result["rank"]}] {result["title"]} '
                f'| score={result["score"]:.4f} '
                f'| {result["chunk_id"]}'
            )
            print(textwrap.shorten(
                result["text"],
                width=500,
                placeholder=" ...",
            ))

    return output

## 3.10 Tests qualitatifs

Nous comparons la réponse produite aux réponses de référence SQuAD.

Les références ne sont pas données au modèle ; elles servent uniquement à
l’évaluation.

In [ ]:
qualitative_indices = [5, 12, 27]

qualitative_results = []

for row_index in qualitative_indices:
    row = dataset[row_index]
    result = answer_with_rag(
        question=row["question"],
        k=DEFAULT_K,
        show_sources=True,
    )

    reference_answer = row["answers"]["text"][0]

    print("\nRÉPONSE DE RÉFÉRENCE")
    print(reference_answer)
    print("\n")

    qualitative_results.append(
        {
            "question": row["question"],
            "reference": reference_answer,
            "rag_answer": result["answer"],
        }
    )

# Exercice 4 — Rôle de BERT dans le retrieval

## 4.1 Fonction de BERT

Dans le retriever dense, BERT transforme :

- chaque chunk en vecteur ;
- chaque question en vecteur dans le même espace.

La recherche sélectionne ensuite les passages dont les vecteurs sont les
plus proches de celui de la question.

## 4.2 Avantages sur TF-IDF

### TF-IDF

TF-IDF valorise les mots rares et les correspondances lexicales.

Avantages :

- rapide ;
- interprétable ;
- peu coûteux ;
- très efficace lorsque les mêmes termes apparaissent.

Limites :

- comprend mal les synonymes ;
- ne modélise pas l’ordre ;
- ne produit pas de représentation contextuelle ;
- dépend fortement du vocabulaire exact.

### BERT dense

Avantages :

- recherche sémantique ;
- représentations contextuelles ;
- meilleure gestion des paraphrases ;
- possibilité de transfert depuis un modèle pré-entraîné.

Limites :

- coût de calcul supérieur ;
- index plus lourd ;
- qualité dépendante du domaine et du fine-tuning ;
- résultats moins directement interprétables.

## 4.3 Influence sur le RAG

Un générateur ne peut pas exploiter une preuve qui n’a pas été retrouvée.
La qualité des embeddings impose donc une limite supérieure au système :

\[
P(\text{bonne réponse})
\leq
P(\text{preuve présente dans le contexte})
\]

Cette relation n’est pas une égalité : même avec la bonne preuve, le
générateur peut encore se tromper.

## 4.4 Baseline TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

tfidf_chunk_matrix = tfidf_vectorizer.fit_transform(chunk_texts)

print("Matrice TF-IDF :", tfidf_chunk_matrix.shape)

In [ ]:
def retrieve_tfidf(question: str, k: int = DEFAULT_K) -> List[Dict]:
    query_vector = tfidf_vectorizer.transform([question])
    scores = linear_kernel(
        query_vector,
        tfidf_chunk_matrix,
    ).ravel()

    top_indices = np.argsort(scores)[::-1][:k]

    results = []
    for rank, index in enumerate(top_indices, start=1):
        chunk = chunks[int(index)]
        results.append(
            {
                "rank": rank,
                "score": float(scores[index]),
                "chunk_index": int(index),
                "chunk_id": chunk.chunk_id,
                "context_id": chunk.context_id,
                "title": chunk.title,
                "text": chunk.text,
                "start_word": chunk.start_word,
                "end_word": chunk.end_word,
            }
        )

    return results

## 4.5 Évaluation Recall@k

Pour chaque question SQuAD, nous connaissons le contexte d’origine.

`Recall@k = 1` pour une question si au moins un des `k` résultats appartient
au contexte attendu.

\[
Recall@k =
\frac{\text{questions dont la preuve est dans le top-k}}
     {\text{nombre total de questions}}
\]

Cette métrique évalue le retriever, pas le générateur.

In [ ]:
evaluation_rows = list(dataset)[:RETRIEVAL_EVAL_SIZE]
evaluation_questions = [
    row["question"] for row in evaluation_rows
]
expected_context_ids = [
    example_to_context_id[row["id"]]
    for row in evaluation_rows
]

max_k = 10

dense_query_embeddings = bert_embedder.encode(
    evaluation_questions,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

dense_scores, dense_indices = faiss_index.search(
    dense_query_embeddings,
    max_k,
)

tfidf_query_matrix = tfidf_vectorizer.transform(
    evaluation_questions
)
tfidf_scores = linear_kernel(
    tfidf_query_matrix,
    tfidf_chunk_matrix,
)
tfidf_indices = np.argsort(
    tfidf_scores,
    axis=1,
)[:, ::-1][:, :max_k]


def recall_at_k(
    ranked_indices: np.ndarray,
    true_context_ids: Sequence[str],
    k: int,
) -> float:
    hits = 0

    for row_index, expected_context_id in enumerate(
        true_context_ids
    ):
        retrieved_context_ids = {
            chunks[int(chunk_index)].context_id
            for chunk_index in ranked_indices[row_index, :k]
        }

        if expected_context_id in retrieved_context_ids:
            hits += 1

    return hits / len(true_context_ids)


metric_rows = []

for k_value in [1, 3, 5, 10]:
    metric_rows.append(
        {
            "method": "BERT dense + FAISS",
            "k": k_value,
            "recall_at_k": recall_at_k(
                dense_indices,
                expected_context_ids,
                k_value,
            ),
        }
    )
    metric_rows.append(
        {
            "method": "TF-IDF",
            "k": k_value,
            "recall_at_k": recall_at_k(
                tfidf_indices,
                expected_context_ids,
                k_value,
            ),
        }
    )

retrieval_metrics = pd.DataFrame(metric_rows)
display(retrieval_metrics)

In [ ]:
pivot_metrics = retrieval_metrics.pivot(
    index="k",
    columns="method",
    values="recall_at_k",
)

pivot_metrics.plot(
    kind="bar",
    figsize=(9, 5),
    ylim=(0, 1.05),
)
plt.title("Comparaison du Recall@k")
plt.ylabel("Recall")
plt.xlabel("k")
plt.xticks(rotation=0)
plt.show()

## 4.6 Interprétation des résultats

Ne concluez pas automatiquement que BERT doit toujours gagner.

TF-IDF peut être supérieur lorsque :

- la question reprend exactement les mots du passage ;
- les noms propres sont déterminants ;
- le corpus est petit et lexicalement stable.

BERT dense peut être supérieur lorsque :

- la question paraphrase le passage ;
- les synonymes remplacent les mots exacts ;
- le sens global est plus important que la correspondance lexicale.

En production, les systèmes performants utilisent souvent un retrieval
hybride :

\[
score =
\alpha \times score_{dense}
+
(1-\alpha) \times score_{sparse}
\]

Ils peuvent ensuite appliquer un reranker plus précis.

# Exercice 5 — RAG contre modèle génératif traditionnel

## 5.1 Architecture

### Modèle génératif seul

```text
Question → connaissances paramétriques du modèle → réponse
```

### RAG

```text
Question → retriever → documents externes
                     ↓
             générateur → réponse
```

## 5.2 Comparaison analytique

| Critère | Générateur seul | RAG |
|---|---|---|
| Source du savoir | paramètres appris | paramètres + documents |
| Mise à jour | réentraînement souvent nécessaire | mise à jour du corpus |
| Citations | difficiles | sources récupérées disponibles |
| Latence | généralement plus faible | retrieval supplémentaire |
| Hallucinations | risque élevé | réduites, mais non supprimées |
| Information privée | absente sans entraînement | corpus privé possible |
| Dépendance au retrieval | aucune | forte |
| Complexité système | plus simple | pipeline plus complexe |

## 5.3 Scénarios où RAG est préférable

- support client fondé sur des procédures internes ;
- assistant juridique avec textes à jour ;
- recherche scientifique avec citations ;
- FAQ d’entreprise ;
- documentation technique évolutive ;
- fact-checking fondé sur un corpus vérifié.

## 5.4 Scénarios où le générateur seul peut suffire

- écriture créative ;
- reformulation ;
- brainstorming ;
- conversation générale sans exigence documentaire ;
- tâches où le style compte davantage que la provenance.

## 5.5 Trade-offs

Le RAG améliore souvent la traçabilité, mais ajoute :

- une base documentaire ;
- un pipeline d’ingestion ;
- une stratégie de chunking ;
- un modèle d’embedding ;
- un index ;
- des métriques ;
- une maintenance des droits d’accès ;
- une surface de sécurité supplémentaire.

## 5.6 Comparaison expérimentale : générateur seul contre RAG

In [ ]:
GENERATOR_ONLY_SYSTEM_MESSAGE = (
    "Answer the user's question concisely. "
    "If you are uncertain, say that you are uncertain."
)


def answer_without_rag(question: str) -> str:
    return generate_with_messages(
        system_message=GENERATOR_ONLY_SYSTEM_MESSAGE,
        user_message=f"Question: {question}\nAnswer:",
        max_new_tokens=100,
    )


comparison_indices = [5, 12, 27]
comparison_rows = []

for row_index in comparison_indices:
    row = dataset[row_index]
    question = row["question"]
    reference = row["answers"]["text"][0]

    no_rag_answer = answer_without_rag(question)
    rag_output = answer_with_rag(
        question,
        k=DEFAULT_K,
        show_sources=False,
    )

    comparison_rows.append(
        {
            "question": question,
            "reference": reference,
            "generator_only": no_rag_answer,
            "rag_answer": rag_output["answer"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

## 5.7 Évaluation simple des réponses

SQuAD utilise traditionnellement Exact Match et token-level F1.

- **Exact Match** exige une égalité après normalisation.
- **F1** mesure le chevauchement entre les tokens.

Ces métriques ont des limites : une paraphrase correcte peut obtenir un
score faible. Elles restent utiles pour une première comparaison.

In [ ]:
def normalize_answer(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = " ".join(text.split())
    return text


def exact_match_score(prediction: str, reference: str) -> float:
    return float(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def token_f1_score(prediction: str, reference: str) -> float:
    prediction_tokens = normalize_answer(prediction).split()
    reference_tokens = normalize_answer(reference).split()

    if not prediction_tokens or not reference_tokens:
        return float(prediction_tokens == reference_tokens)

    prediction_counts = defaultdict(int)
    reference_counts = defaultdict(int)

    for token in prediction_tokens:
        prediction_counts[token] += 1

    for token in reference_tokens:
        reference_counts[token] += 1

    common = sum(
        min(prediction_counts[token], reference_counts[token])
        for token in prediction_counts
    )

    if common == 0:
        return 0.0

    precision = common / len(prediction_tokens)
    recall = common / len(reference_tokens)

    return 2 * precision * recall / (precision + recall)


scored_rows = []

for row in comparison_rows:
    for method_name, answer_key in [
        ("Generator only", "generator_only"),
        ("RAG", "rag_answer"),
    ]:
        scored_rows.append(
            {
                "method": method_name,
                "question": row["question"],
                "exact_match": exact_match_score(
                    row[answer_key],
                    row["reference"],
                ),
                "token_f1": token_f1_score(
                    row[answer_key],
                    row["reference"],
                ),
            }
        )

answer_scores = pd.DataFrame(scored_rows)

display(answer_scores)
display(
    answer_scores.groupby("method")[
        ["exact_match", "token_f1"]
    ].mean()
)

> L’échantillon de trois questions est trop petit pour une conclusion
> statistique. Il sert à illustrer la méthode. Une évaluation sérieuse
> utiliserait davantage de questions, plusieurs références et une analyse
> humaine de la fidélité aux sources.

# Exercice 6 — Futur du RAG en NLP

## 6.1 Évolution du RAG

Le RAG initial associe une mémoire paramétrique à un index documentaire
externe. Les travaux récents cherchent surtout à améliorer quatre points :

1. décider **quand** récupérer ;
2. récupérer des preuves réellement pertinentes ;
3. organiser le contexte ;
4. vérifier que la génération respecte les preuves.

## 6.2 Retrieval adaptatif et auto-réflexion

**Self-RAG** apprend à déclencher le retrieval lorsque nécessaire et à
critiquer les passages et la réponse avec des tokens de réflexion.

Opportunité :

- éviter le retrieval inutile ;
- améliorer les citations ;
- adapter le comportement à la tâche.

Difficulté :

- entraînement plus complexe ;
- coût d’inférence supérieur ;
- évaluation de la réflexion.

## 6.3 Corrective RAG

**CRAG** ajoute un évaluateur de retrieval. Lorsque les documents sont
insuffisants, le système peut :

- corriger la recherche ;
- filtrer le bruit ;
- consulter une source externe ;
- recomposer le contexte.

L’idée importante est que le pipeline ne doit pas supposer que le premier
retrieval est correct.

## 6.4 Retrieval hiérarchique

**RAPTOR** construit un arbre de chunks et de résumés.

Il permet de rechercher :

- des détails locaux ;
- des résumés de sections ;
- des informations globales.

Cette approche est adaptée aux longs documents et aux questions nécessitant
plusieurs niveaux d’abstraction.

## 6.5 GraphRAG

Le retrieval vectoriel classique traite souvent les chunks comme des unités
indépendantes. GraphRAG représente aussi :

- les entités ;
- leurs relations ;
- les communautés ;
- les chemins multi-hop.

Applications :

- analyse de réseaux ;
- dossiers complexes ;
- connaissance scientifique ;
- investigation ;
- intelligence économique.

## 6.6 Retrieval hybride et reranking

Une tendance forte consiste à combiner :

- BM25 ou TF-IDF pour les correspondances exactes ;
- embeddings denses pour la sémantique ;
- filtres de métadonnées ;
- rerankers cross-encoder ;
- règles métier.

Le retriever initial vise le rappel. Le reranker améliore ensuite la
précision.

## 6.7 RAG multimodal

Les futurs systèmes ne rechercheront pas seulement du texte, mais aussi :

- images ;
- tableaux ;
- audio ;
- vidéo ;
- code ;
- graphes ;
- données structurées.

Le défi consiste à aligner ces modalités dans une représentation exploitable
et à fournir une provenance compréhensible.

## 6.8 RAG agentique

Un agent peut :

1. reformuler la question ;
2. choisir une source ;
3. effectuer plusieurs recherches ;
4. vérifier les contradictions ;
5. appeler des outils ;
6. produire une réponse avec preuves.

Cette flexibilité augmente également les risques de boucle, de coût et
d’actions incorrectes.

## 6.9 Scalabilité et efficacité

Les défis comprennent :

- des milliards de chunks ;
- la latence ;
- la mémoire ;
- les mises à jour incrémentales ;
- la déduplication ;
- les index distribués ;
- le cache ;
- le routage entre plusieurs collections.

Des index approximatifs, la quantification et les modèles d’embedding plus
petits peuvent réduire le coût.

## 6.10 Évaluation

Évaluer seulement la réponse finale est insuffisant.

Il faut mesurer :

- Recall@k ;
- Precision@k ;
- MRR ou nDCG ;
- fidélité au contexte ;
- exactitude des citations ;
- complétude ;
- robustesse au bruit ;
- latence et coût ;
- comportement lorsque les sources se contredisent.

## 6.11 Sécurité et éthique

Un système RAG peut exposer ou amplifier :

- des données privées ;
- des biais du corpus ;
- des documents obsolètes ;
- des contenus malveillants ;
- du prompt injection dans les documents ;
- des violations de droits d’accès ;
- des citations trompeuses.

Les protections nécessaires incluent :

- contrôle d’accès avant retrieval ;
- filtrage des données sensibles ;
- traçabilité ;
- séparation instructions/données ;
- validation des sources ;
- politiques de rétention ;
- audits réguliers.

## 6.12 Opportunités

Le RAG peut transformer :

- les assistants d’entreprise ;
- la recherche documentaire ;
- la médecine et le droit, avec supervision ;
- l’éducation personnalisée ;
- le support technique ;
- la veille scientifique ;
- la génération de contenu sourcé.

Son potentiel dépend moins de la taille du générateur que de la qualité de
toute la chaîne documentaire.

# Méthode académique de débogage

## Étape 1 — Vérifier les données

- le texte est-il propre ?
- les métadonnées sont-elles présentes ?
- le corpus contient-il la réponse ?
- existe-t-il des doublons ou versions obsolètes ?

## Étape 2 — Vérifier le chunking

- la réponse est-elle coupée entre deux chunks ?
- les chunks sont-ils trop longs ?
- l’overlap crée-t-il trop de redondance ?

## Étape 3 — Vérifier le retrieval

- le bon contexte apparaît-il dans top-1, top-3 ou top-5 ?
- BERT et TF-IDF échouent-ils sur les mêmes questions ?
- les scores sont-ils proches ou nettement séparés ?

## Étape 4 — Vérifier le prompt

- le modèle sait-il qu’il doit se limiter au contexte ?
- la question est-elle clairement séparée des documents ?
- les sources sont-elles identifiables ?

## Étape 5 — Vérifier la génération

- la bonne preuve est-elle présente mais ignorée ?
- le modèle combine-t-il des passages incompatibles ?
- reconnaît-il l’absence de réponse ?

## Étape 6 — Mesurer

- Recall@k pour le retrieval ;
- Exact Match et F1 pour la réponse ;
- fidélité et citations par évaluation humaine ;
- temps d’indexation et d’inférence.

# Conclusion générale

Les modèles séquentiels traditionnels ont du mal à maintenir des relations
sur de longues distances. L’attention réduit cette difficulté en permettant
des interactions directes entre tokens.

BERT exploite l’attention bidirectionnelle pour produire des représentations
contextuelles. Dans un RAG, ces représentations rendent possible une
recherche sémantique plus riche que la seule correspondance lexicale.

Le RAG complète un générateur avec une mémoire externe. Il facilite la mise
à jour des connaissances et la traçabilité, mais introduit de nouvelles
dépendances : qualité des données, chunking, retrieval, sécurité et
évaluation.

La leçon principale est la suivante :

> Un bon générateur ne compense pas un mauvais retrieval, et un bon
> retrieval ne garantit pas à lui seul une réponse fidèle.

# Références principales

1. Vaswani, A. et al. (2017). *Attention Is All You Need*.
   https://arxiv.org/abs/1706.03762

2. Devlin, J. et al. (2018). *BERT: Pre-training of Deep Bidirectional
   Transformers for Language Understanding*.
   https://arxiv.org/abs/1810.04805

3. Karpukhin, V. et al. (2020). *Dense Passage Retrieval for Open-Domain
   Question Answering*.
   https://arxiv.org/abs/2004.04906

4. Lewis, P. et al. (2020). *Retrieval-Augmented Generation for
   Knowledge-Intensive NLP Tasks*.
   https://arxiv.org/abs/2005.11401

5. Asai, A. et al. (2023). *Self-RAG: Learning to Retrieve, Generate, and
   Critique through Self-Reflection*.
   https://arxiv.org/abs/2310.11511

6. Yan, S.-Q. et al. (2024). *Corrective Retrieval Augmented Generation*.
   https://arxiv.org/abs/2401.15884

7. Sarthi, P. et al. (2024). *RAPTOR: Recursive Abstractive Processing for
   Tree-Organized Retrieval*.
   https://arxiv.org/abs/2401.18059

8. Hu, Y. et al. (2024). *GRAG: Graph Retrieval-Augmented Generation*.
   https://arxiv.org/abs/2405.16506

9. Douze, M. et al. (2024). *The Faiss Library*.
   https://arxiv.org/abs/2401.08281

10. Sharma, C. (2025). *Retrieval-Augmented Generation: A Comprehensive
    Survey of Architectures, Enhancements, and Robustness Frontiers*.
    https://arxiv.org/abs/2506.00054

11. Yang, A. et al. (2024). *Qwen2.5 Technical Report*.
    https://arxiv.org/abs/2412.15115